# Pipeline Completo: Proyección de Exportaciones por Industria

Este notebook realiza todo el proceso desde los datos iniciales hasta la gráfica final de proyecciones por industria manteniendo proporciones constantes.

## Inputs Requeridos

1. **`indices_normalizados_20anios.xlsx`** con dos hojas:
   - `Indices_Normalizados`: columnas [periodo, indice_normalizado]
   - `Industrias`: columnas [CIIU, Share]

## Outputs Generados

1. **`proyecciones_por_industria.xlsx`** con tres hojas:
   - `Proyecciones_Industrias`: valores por industria y período
   - `Resumen_Total`: índice normalizado y valor total agregado
   - `Proporciones_Base`: proporciones constantes usadas
2. **`grafica_apilada_sin_otros.png`**: visualización de área apilada

## Paso 0: Importar Librerías

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Configuración de visualización
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

print("✓ Librerías importadas exitosamente")

## Paso 1: Cargar Datos de Entrada

In [ ]:
# Archivo de entrada
input_file = 'indices_normalizados_20anios.xlsx'

# Cargar índices normalizados y proporciones de industrias
xl_file = pd.ExcelFile(input_file)
indices = pd.read_excel(xl_file, sheet_name='Indices_Normalizados')
proporciones = pd.read_excel(xl_file, sheet_name='Industrias')

print(f"✓ Índices cargados: {len(indices)} períodos")
print(f"✓ Industrias cargadas: {len(proporciones)} sectores")
print(f"✓ Total participación: {proporciones['Share'].sum():.4f}")

# Mostrar primeras filas
print("\nÍndices Normalizados (primeras 5 filas):")
display(indices.head())

print("\nProporciones de Industrias:")
display(proporciones)

## Paso 2: Definir Parámetros de Proyección

Aquí defines los valores inicial y final de tu proyección. Estos valores determinarán el rango de crecimiento de las exportaciones.

In [ ]:
# PARÁMETROS: Ajusta estos valores según tu proyección
VALOR_INICIAL = 0    # Valor de exportaciones en período 0
VALOR_FINAL = 100    # Valor de exportaciones en período 19

print(f"Valor inicial (A): {VALOR_INICIAL}")
print(f"Valor final (B): {VALOR_FINAL}")
print(f"Crecimiento total: {VALOR_FINAL - VALOR_INICIAL}")

## Paso 3: Aplicar Fórmula de Reescalado

La fórmula de reescalado es:

$$\text{Proyección}(t) = A + (B - A) \times \frac{\text{Índice}(t) - \text{Índice}(0)}{\text{Índice}(19) - \text{Índice}(0)}$$

Esta fórmula distribuye el crecimiento total siguiendo la dinámica temporal del índice histórico.

In [ ]:
# Extraer índices inicial y final
indice_inicial = indices.loc[indices['periodo'] == 0, 'indice_normalizado'].values[0]
indice_final = indices.loc[indices['periodo'] == 19, 'indice_normalizado'].values[0]

print(f"Índice inicial: {indice_inicial:.4f}")
print(f"Índice final: {indice_final:.4f}")
print(f"Factor de crecimiento del índice: {indice_final:.2f}x")

# Aplicar fórmula de reescalado
indices['valor_total'] = VALOR_INICIAL + (VALOR_FINAL - VALOR_INICIAL) * \
                         ((indices['indice_normalizado'] - indice_inicial) / 
                          (indice_final - indice_inicial))

print(f"\n✓ Valores totales calculados para {len(indices)} períodos")

# Mostrar resultado
display(indices)

## Paso 4: Validar Proporciones de Industrias

Las proporciones de todas las industrias deben sumar exactamente 100% (1.0). Si no es así, se normalizan automáticamente.

In [ ]:
# Verificar que las proporciones sumen 100%
total_share = proporciones['Share'].sum()
print(f"Total de proporciones: {total_share:.6f}")

if abs(total_share - 1.0) > 0.01:
    print(f"⚠️  ADVERTENCIA: Las proporciones suman {total_share*100:.2f}%, no 100%")
    print("Normalizando proporciones...")
    proporciones['Share'] = proporciones['Share'] / total_share
    print(f"✓ Proporciones normalizadas, nuevo total: {proporciones['Share'].sum():.6f}")
else:
    print("✓ Proporciones válidas (suman 100%)")

## Paso 5: Calcular Proyecciones por Industria

Para cada período, se calcula el valor de cada industria multiplicando el valor total por su proporción (share). Esto mantiene las proporciones constantes a lo largo del tiempo.

In [ ]:
# Crear matriz de proyecciones: cada fila = período, cada columna = industria
proyecciones_industrias = []

for _, row_periodo in indices.iterrows():
    periodo = int(row_periodo['periodo'])
    valor_total = row_periodo['valor_total']
    
    # Diccionario para este período
    valores_industrias = {'periodo': periodo, 'valor_total': valor_total}
    
    # Calcular valor de cada industria = valor_total × share
    for _, row_industria in proporciones.iterrows():
        ciiu = str(row_industria['CIIU'])
        share = row_industria['Share']
        valor_industria = valor_total * share
        valores_industrias[f'CIIU_{ciiu}'] = valor_industria
    
    proyecciones_industrias.append(valores_industrias)

# Convertir a DataFrame
df_proyecciones = pd.DataFrame(proyecciones_industrias)

print(f"✓ Proyecciones calculadas: {df_proyecciones.shape[0]} períodos × {df_proyecciones.shape[1]-2} industrias")

# Mostrar primeras filas
print("\nPrimeras 5 filas de proyecciones:")
display(df_proyecciones.head())

## Paso 6: Validar Coherencia

Verificamos que la suma de todas las industrias en cada período sea igual al valor total proyectado. Diferencias menores a 1e-10 son aceptables (errores de redondeo).

In [ ]:
# Verificar que suma de industrias = valor total en cada período
df_proyecciones['suma_verificacion'] = df_proyecciones.filter(like='CIIU_').sum(axis=1)
df_proyecciones['diferencia'] = df_proyecciones['valor_total'] - df_proyecciones['suma_verificacion']

max_diferencia = df_proyecciones['diferencia'].abs().max()
print(f"Diferencia máxima (error de redondeo): {max_diferencia:.10f}")

if max_diferencia < 1e-10:
    print("✓ Validación exitosa: suma de industrias = total en todos los períodos")
else:
    print("⚠️  ADVERTENCIA: Diferencias mayores a tolerancia encontradas")

# Mostrar muestra de verificación
print("\nMuestra de verificación (primeros 10 períodos):")
display(df_proyecciones[['periodo', 'valor_total', 'suma_verificacion', 'diferencia']].head(10))

## Paso 7: Exportar Resultados a Excel

In [ ]:
output_excel = 'proyecciones_por_industria.xlsx'

with pd.ExcelWriter(output_excel, engine='openpyxl') as writer:
    # Hoja 1: Proyecciones por industria (limpiar columnas de verificación)
    df_proyecciones.drop(columns=['suma_verificacion', 'diferencia']).to_excel(
        writer, sheet_name='Proyecciones_Industrias', index=False
    )
    
    # Hoja 2: Resumen con índice y valor total
    indices.to_excel(writer, sheet_name='Resumen_Total', index=False)
    
    # Hoja 3: Proporciones base originales
    proporciones.to_excel(writer, sheet_name='Proporciones_Base', index=False)

print(f"✓ Archivo Excel exportado: {output_excel}")
print("\nHojas incluidas:")
print("  1. Proyecciones_Industrias: Valor por industria en cada período")
print("  2. Resumen_Total: Índice normalizado y valor total agregado")
print("  3. Proporciones_Base: Proporciones constantes usadas")

## Paso 8: Generar Gráfica sin "Otros"

Creamos una visualización de área apilada mostrando solo las industrias específicas, excluyendo la categoría "Otros".

In [ ]:
# Recargar datos para gráfica
df_grafica = pd.read_excel(output_excel, sheet_name='Proyecciones_Industrias')

# Excluir "Otros" y preparar datos
industrias_sin_otros = [col for col in df_grafica.columns if col not in ['periodo', 'Otros']]

# Ordenar por valor final (mayor a menor)
valores_finales_sin_otros = {col: df_grafica[col].iloc[-1] for col in industrias_sin_otros}
industrias_ordenadas = sorted(valores_finales_sin_otros.items(), 
                             key=lambda x: x[1], reverse=True)

print(f"✓ Industrias a graficar: {len(industrias_ordenadas)}")

# Paleta de colores distinguibles
colores = [
    '#e6194b',  # Rojo brillante
    '#3cb44b',  # Verde
    '#ffe119',  # Amarillo
    '#4363d8',  # Azul
    '#f58231',  # Naranja
    '#911eb4',  # Púrpura
    '#46f0f0',  # Cian
    '#f032e6',  # Magenta
    '#bcf60c',  # Lima
    '#fabebe',  # Rosa claro
]

# Preparar datos para gráfica de área apilada
industrias_data = []
labels = []

for idx, (industria, valor) in enumerate(industrias_ordenadas):
    industrias_data.append(df_grafica[industria].values)
    labels.append(industria)

# Crear figura
fig, ax = plt.subplots(figsize=(12, 11))

ax.stackplot(df_grafica['periodo'], *industrias_data, 
            labels=labels, 
            colors=colores[:len(industrias_data)],
            alpha=0.9)

ax.set_xlabel('Períodos', fontsize=18, fontweight='bold')
ax.set_ylabel('Proporción de ganancia exportadora', fontsize=18, fontweight='bold')
ax.set_title('Proyección de Exportaciones por Industria Específica (Sin Otros)', 
            fontsize=22, fontweight='bold', pad=25)
ax.legend(loc='upper center', fontsize=11, ncol=2, framealpha=0.95,
         bbox_to_anchor=(0.5, -0.12))
ax.grid(True, alpha=0.3, axis='y', linestyle='--', linewidth=0.8)
ax.set_xlim(0, 19)
ax.set_xticks(range(0, 20, 2))
ax.tick_params(axis='both', which='major', labelsize=14)

plt.tight_layout()

output_grafica = 'grafica_apilada_sin_otros.png'
plt.savefig(output_grafica, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✓ Gráfica exportada: {output_grafica}")

## Paso 9: Resumen Final

Resumen de estadísticas clave y ranking de las principales industrias.

In [ ]:
print("="*80)
print("RESUMEN FINAL")
print("="*80)

print(f"\nPERÍODOS: {len(indices)}")
print(f"INDUSTRIAS ESPECÍFICAS: {len(industrias_sin_otros)}")
print(f"RANGO DE PROYECCIÓN: {VALOR_INICIAL} → {VALOR_FINAL}")

print("\nTOP 5 INDUSTRIAS (por valor final):")
for idx, (industria, valor) in enumerate(industrias_ordenadas[:5], 1):
    porcentaje = (valor / VALOR_FINAL) * 100
    print(f"  {idx}. {industria:<60} | {valor:>6.2f} ({porcentaje:>5.2f}%)")

print("\n" + "="*80)
print("PIPELINE COMPLETADO EXITOSAMENTE")
print("="*80)
print("\nARCHIVOS GENERADOS:")
print(f"  1. {output_excel}")
print(f"  2. {output_grafica}")